# 01 · Exploración y transformación — de Bronze a Silver

**Preguntas que se contestan en esta capa:** ¿cuántas líneas de pedido son válidas? ¿qué problemas de calidad tenían los datos y cuántas filas afectaban cada uno?

## Qué pasa en este notebook

Silver es la capa donde los datos dejan de ser "tal como llegaron" y pasan a ser confiables. El trabajo sigue el mismo orden que se vio en el curso (Módulo 2): **primero se mide el problema y después se limpia**. Nunca se limpia a ciegas.

1. **Perfil de calidad.** Se cuentan nulos, duplicados y valores fuera de rango en Bronze. Ese perfil define qué reglas de limpieza necesita cada tabla.
2. **Limpieza y tipado.**
   - `productos`: las tallas se normalizan a `XS, S, M, L, XL`; los colores nulos se marcan como `Sin dato`; el precio, que venía como texto con `$`, se convierte a número.
   - `clientes`: las ciudades nulas pasan a `Desconocida`; se descartan el nombre y el correo, porque el análisis no los necesita y así los datos personales se quedan solo en Bronze.
   - `lineas_pedido`: se eliminan duplicados por `linea_id`, se descartan las filas sin cliente o producto y las de cantidad cero, y el precio, que venía como texto con coma decimal, se convierte a número.
3. **Etiqueta todavía en cero.** Cada línea entra a Silver con `fue_devuelto = 0` y `fecha_devolucion` nula, porque las devoluciones aún no han llegado. Se aplican en el siguiente notebook con un `MERGE INTO`.

**En esta celda** se hereda la configuración compartida (catálogo, `tbl()`, `escribir_delta()` y las reglas de negocio).

In [ ]:
%run ./00_Setup/00_Configuracion

**En esta celda** se importan las funciones y tipos de PySpark que usa el resto del notebook.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql.functions import broadcast

**En esta celda** se calcula el perfil de calidad de `lineas_pedido` en Bronze con tres técnicas: contar nulos por columna en una sola pasada (`F.sum(col.isNull().cast(int))`), comparar el total de filas contra los identificadores distintos para detectar duplicados, y contar las filas que violan un rango válido con `filter()`.

In [ ]:
df_lineas_bronze = spark.table(tbl(SCHEMA_BRONZE, "lineas_pedido"))
total_lineas = df_lineas_bronze.count()

print("PERFIL DE CALIDAD: lineas_pedido (Bronze)")
print("=" * 60)
print(f"Total de registros: {total_lineas:,}")

print("\n1. Nulos por columna:")
display(df_lineas_bronze.select([
    F.sum(F.col(c).isNull().cast(IntegerType())).alias(c)
    for c in ["cliente_id", "producto_id", "cantidad", "precio_pagado"]
]))

print("\n2. Duplicados por clave natural (linea_id):")
unicos = df_lineas_bronze.select("linea_id").distinct().count()
print(f"   Registros: {total_lineas:,}  |  linea_id unicos: {unicos:,}  |  Duplicados: {total_lineas - unicos:,}")

print("\n3. Valores fuera de rango y formatos:")
n_cantidad = df_lineas_bronze.filter("cantidad <= 0").count()
n_coma = df_lineas_bronze.filter("precio_pagado LIKE '%,%'").count()
print(f"   cantidad <= 0                  : {n_cantidad:,}")
print(f"   precio con coma decimal (texto): {n_coma:,}")

**En esta celda** se calcula el perfil de `productos` y `clientes`: cómo están escritas las tallas (para ver cuántas variantes hay), cuántos colores y ciudades son nulos y cuántos precios traen el signo `$`.

In [ ]:
df_productos_bronze = spark.table(tbl(SCHEMA_BRONZE, "productos"))
df_clientes_bronze  = spark.table(tbl(SCHEMA_BRONZE, "clientes"))

print("PERFIL DE CALIDAD: productos y clientes (Bronze)")
print("=" * 60)
print("Variantes de talla encontradas:")
display(df_productos_bronze.groupBy("talla").count().orderBy("talla"))

n_color_nulo  = df_productos_bronze.filter("color IS NULL").count()
n_precio_signo = df_productos_bronze.filter("precio_lista LIKE '$%'").count()
n_ciudad_nula = df_clientes_bronze.filter("ciudad IS NULL").count()
print(f"productos con color nulo      : {n_color_nulo:,}")
print(f"productos con precio con '$'  : {n_precio_signo:,}")
print(f"clientes con ciudad nula      : {n_ciudad_nula:,}")

**En esta celda** se limpia `productos` y se escribe `silver.productos`. La talla se pasa a mayúsculas y sin espacios y se traducen los nombres largos (`MEDIUM` → `M`) con una cascada de `F.when()`. El precio se limpia con `regexp_replace` (se quita todo lo que no sea dígito o punto) y se convierte con `try_cast`, que devuelve nulo en vez de fallar si algún valor no es numérico.

In [ ]:
talla_limpia = F.upper(F.trim(F.col("talla")))

df_productos_silver = (
    df_productos_bronze
    .dropDuplicates(["producto_id"])
    .withColumn("talla",
        F.when(talla_limpia == "EXTRA SMALL", "XS")
         .when(talla_limpia == "SMALL", "S")
         .when(talla_limpia == "MEDIUM", "M")
         .when(talla_limpia == "LARGE", "L")
         .when(talla_limpia == "EXTRA LARGE", "XL")
         .otherwise(talla_limpia))
    .withColumn("color", F.coalesce(F.col("color"), F.lit("Sin dato")))
    .withColumn("precio_lista", F.expr("try_cast(regexp_replace(precio_lista, '[^0-9.]', '') AS DOUBLE)"))
    .select("producto_id", "nombre", "categoria", "talla", "color", "precio_lista")
)

escribir_delta(df_productos_silver, SCHEMA_SILVER, "productos", "Productos limpios: talla normalizada, precio numerico")
display(df_productos_silver.groupBy("talla").count().orderBy("talla"))

**En esta celda** se limpia `clientes` y se escribe `silver.clientes`. Las ciudades nulas pasan a `Desconocida`, y se dejan fuera el nombre y el correo: son datos personales que el análisis no necesita, así que solo existen en Bronze.

In [ ]:
df_clientes_silver = (
    df_clientes_bronze
    .dropDuplicates(["cliente_id"])
    .withColumn("ciudad", F.coalesce(F.col("ciudad"), F.lit("Desconocida")))
    .select("cliente_id", "ciudad", "segmento", "fecha_registro")
)

escribir_delta(df_clientes_silver, SCHEMA_SILVER, "clientes", "Clientes sin datos personales (sin nombre ni correo)")

**En esta celda** se limpia `lineas_pedido` y se escribe `silver.lineas_pedido`. Se elimina cada duplicado por `linea_id`, se descartan las filas con cliente o producto nulo o con cantidad menor o igual a cero, y el precio (que venía con coma decimal) se convierte a número. Cada línea entra con `fue_devuelto = 0` y `fecha_devolucion` nula, porque la devolución aún no ha llegado.

**Por qué el `mes` como partición:** `fecha` tiene 365 valores y generaría 365 carpetas con unas 55 filas cada una. `mes` tiene 12 valores, así que cada partición tiene suficientes datos.

In [ ]:
df_lineas_silver = (
    df_lineas_bronze
    .dropDuplicates(["linea_id"])
    .filter(F.col("cliente_id").isNotNull() & F.col("producto_id").isNotNull() & (F.col("cantidad") > 0))
    .withColumn("precio_pagado", F.expr("try_cast(regexp_replace(precio_pagado, ',', '.') AS DOUBLE)"))
    .filter(F.col("precio_pagado") > 0)
    .withColumn("fue_devuelto", F.lit(0))                                  # todavia no llegan las devoluciones
    .withColumn("fecha_devolucion", F.lit(None).cast(DateType()))
    .select("linea_id", "pedido_id", "cliente_id", "producto_id", "fecha", "mes", "cantidad",
            "descuento_pct", "precio_pagado", "fue_devuelto", "fecha_devolucion")
)

escribir_delta(df_lineas_silver, SCHEMA_SILVER, "lineas_pedido",
               "Lineas de pedido limpias. fue_devuelto se actualiza con MERGE", partition_col="mes")

**En esta celda** se compara Bronze contra Silver para responder la pregunta de esta capa: cuántas filas se descartaron y por qué. Es la evidencia de que la separación en capas tiene sentido, porque Bronze conserva los datos originales y Silver conserva solo los válidos.

In [ ]:
bronze_n = spark.table(tbl(SCHEMA_BRONZE, "lineas_pedido")).count()
silver_n = spark.table(tbl(SCHEMA_SILVER, "lineas_pedido")).count()

print("=" * 60)
print("BRONZE vs SILVER — lineas_pedido")
print("=" * 60)
print(f"  Bronze (tal como llego)      : {bronze_n:,}")
print(f"  Silver (limpias y validas)   : {silver_n:,}")
print(f"  Filas descartadas            : {bronze_n - silver_n:,} ({(bronze_n - silver_n) / bronze_n:.1%})")
print("  Motivos: duplicados por linea_id, cliente o producto nulo, cantidad <= 0.")
print("\nLas devoluciones todavia no se aplicaron: fue_devuelto = 0 en todas las lineas.")
display(spark.table(tbl(SCHEMA_SILVER, "lineas_pedido")).groupBy("fue_devuelto").count())

Silver · Transformación &nbsp;·&nbsp; Siguiente: **02_Delta_Lake**